In [3]:
import pandas as pd
import numpy as np

In [4]:
## load data

data = pd.read_csv(r'D:\weq-preview\madatory task 1\train.csv')

In [5]:
X = data['title']
y = data['label_group']

### Experiment 1 using IF-IDF

In [6]:
import re
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\krish\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [7]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

In [8]:
X_list_corpus = list(X)

In [7]:
len(X_list_corpus)

34250

In [8]:
for i in range(len(X_list_corpus)):
    review = re.sub('[^a-zA-Z]',' ',X_list_corpus[i])
    review = review.lower()
    review = review.split()
    review = [lemmatizer.lemmatize(word.lower(),pos='v') for word in review if word not in set(stopwords.words('english'))]
    review = ' '.join(review)
    X_list_corpus[i] = review
    ## takes 5 minitues atleast

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000


In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer
tdidf = TfidfVectorizer(max_features=2500, ngram_range=(1,2))
X_tfidf = tdidf.fit_transform(X_list_corpus).toarray()

In [29]:
X_tfidf.shape

(34250, 2500)

In [10]:
import numpy as np
np.set_printoptions(edgeitems=30,linewidth=100000,formatter=dict(float=lambda x: '%.3g' % x))

In [11]:
tdidf.vocabulary_

{'paper': np.int64(1606),
 'bag': np.int64(136),
 'secret': np.int64(1940),
 'double': np.int64(589),
 'tape': np.int64(2205),
 'mm': np.int64(1421),
 'original': np.int64(1559),
 'foam': np.int64(703),
 'double tape': np.int64(590),
 'gr': np.int64(810),
 'daster': np.int64(529),
 'batik': np.int64(193),
 'lengan': np.int64(1224),
 'pendek': np.int64(1648),
 'motif': np.int64(1440),
 'leher': np.int64(1218),
 'kancing': np.int64(1040),
 'karakter': np.int64(1054),
 'lengan pendek': np.int64(1226),
 'xc': np.int64(2461),
 'ml': np.int64(1417),
 'celana': np.int64(395),
 'wanita': np.int64(2411),
 'bb': np.int64(206),
 'kg': np.int64(1098),
 'bisa': np.int64(262),
 'cod': np.int64(448),
 'bisa cod': np.int64(264),
 'anak': np.int64(61),
 'size': np.int64(2034),
 'thn': np.int64(2255),
 'kulot': np.int64(1164),
 'plisket': np.int64(1714),
 'salur': np.int64(1896),
 'candy': np.int64(369),
 'premium': np.int64(1745),
 'tempelan': np.int64(2234),
 'kulkas': np.int64(1163),
 'magnet': np.in

In [13]:
##

In [13]:
from collections import defaultdict

label_groups = defaultdict(list)

for i, label in enumerate(y):
    label_groups[label].append(i)

print("Number of groups:", len(label_groups))

Number of groups: 11014


In [14]:
from itertools import combinations
import random

random.seed(42)

positive_pairs = []

for label, indices in label_groups.items():
    if len(indices) >= 2:
        group_pairs = list(combinations(indices, 2))
        random.shuffle(group_pairs)

        for i, j in group_pairs:
            positive_pairs.append((i, j, 1))

            if len(positive_pairs) >= 30000:
                break

    if len(positive_pairs) >= 30000:
        break

print("Positive pairs:", len(positive_pairs))

Positive pairs: 30000


In [15]:
negative_pairs = []

n = len(y)

while len(negative_pairs) < 30000:
    i, j = random.sample(range(n), 2)

    if y.iloc[i] != y.iloc[j]:
        negative_pairs.append((i, j, 0))

print("Negative pairs:", len(negative_pairs))

Negative pairs: 30000


In [16]:
pairs = positive_pairs + negative_pairs

random.shuffle(pairs)

pairs_df = pd.DataFrame(
    pairs,
    columns=["index_1", "index_2", "match"]
)

print(pairs_df.shape)
print(pairs_df["match"].value_counts())

(60000, 3)
match
0    30000
1    30000
Name: count, dtype: int64


In [53]:
pairs_df

,index_1,index_2,match
0,5312,1168,0
1,578,16511,1
2,146,3769,1
3,5547,18150,0
4,13738,18558,1
...,...,...,...
59995,12509,30057,0
59996,732,17908,1
59997,5424,25177,0
59998,20418,12619,0


In [54]:
cosine_sim = []

In [55]:
## Cosine similarity checks

from sklearn.metrics.pairwise import cosine_similarity

pairs_df['similarity'] = pairs_df.apply(
    lambda row: cosine_similarity([X_tfidf[row['index_1']]],[X_tfidf[row['index_2']]])[0][0],axis=1
)

In [56]:
threshold = 0.5

pairs_df["prediction"] = (
    pairs_df["similarity"] >= threshold
).astype(int)

In [19]:
# Store the first 10,000 pairs
existing_pairs = set(
    zip(pairs_df["index_1"], pairs_df["index_2"])
)

print("Existing pairs:", len(existing_pairs))

Existing pairs: 60000


In [20]:
second_pairs = []

while len(second_pairs) < 10000:
    i, j = random.sample(range(n), 2)

    # Keep order consistent
    pair = (min(i, j), max(i, j))

    # Make sure this pair was NOT in the first 10,000
    if pair not in existing_pairs:
        
        if y.iloc[i] == y.iloc[j]:
            match = 1
        else:
            match = 0

        second_pairs.append((pair[0], pair[1], match))
        existing_pairs.add(pair)

second_pairs = pd.DataFrame(
    second_pairs,
    columns=["index_1", "index_2", "match"]
)


In [59]:
## Cosine similarity checks

from sklearn.metrics.pairwise import cosine_similarity

second_pairs['similarity'] = second_pairs.apply(
    lambda row: cosine_similarity([X_tfidf[row['index_1']]],[X_tfidf[row['index_2']]])[0][0],axis=1
)

In [60]:
threshold = 0.5

second_pairs["prediction"] = (
    second_pairs["similarity"] >= threshold
).astype(int)

In [61]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_true = second_pairs["match"]
y_pred = second_pairs["prediction"]

print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall   :", recall_score(y_true, y_pred))
print("F1-score :", f1_score(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Accuracy : 0.9995
Precision: 0.0
Recall   : 0.0
F1-score : 0.0

Confusion Matrix:
[[9995    5]
 [   0    0]]


c:\Users\krish\anaconda3\envs\pytorch_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [65]:
## Apply on testing data 
## load data

data_test = pd.read_csv(r'D:\weq-preview\madatory task 1\test.csv')
data_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   posting_id   3 non-null      str  
 1   image        3 non-null      str  
 2   image_phash  3 non-null      str  
 3   title        3 non-null      str  
dtypes: str(4)
memory usage: 637.0 bytes


In [62]:
X_list_test = list(data_test['title'])

for i in range(len(X_list_test)):
    review = re.sub('[^a-zA-Z]',' ',X_list_test[i])
    review = review.lower()
    review = review.split()
    review = [lemmatizer.lemmatize(word.lower(),pos='v') for word in review if word not in set(stopwords.words('english'))]
    review = ' '.join(review)
    X_list_test[i] = review
    ## takes 5 minitues atleast


X_test_tfidf = tdidf.transform(X_list_test).toarray()

In [82]:
X_test_tfidf

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.47, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..., 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..., 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..., 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [83]:
matches = []

for i in range(len(X_test_tfidf)):
    for j in range(i + 1, len(X_test_tfidf)):

        similarity = cosine_similarity(
            X_test_tfidf[i].reshape(1, -1),
            X_test_tfidf[j].reshape(1, -1)
        )[0][0]

        if similarity >= 0.5:
            matches.append({
                "posting_id": data_test["posting_id"].iloc[i],
                "matches": data_test["posting_id"].iloc[j],
                "similarity": similarity
            })

sub = pd.DataFrame(matches)

print(sub.head())
print("Number of matches:", len(sub))

Empty DataFrame
Columns: []
Index: []
Number of matches: 0


### Experiment 2 using Word2vec

In [9]:
from gensim.models import Word2Vec

# X_list_corpus is already cleaned (lowercase, lemmatized, stopwords removed)
tokenized = [t.split() for t in X_list_corpus]

w2v = Word2Vec(
    sentences=tokenized,
    vector_size=100,   # embedding dimension
    window=5,
    min_count=2,       # ignore words seen only once
    sg=1,              # skip-gram works better on small/short-text corpora
    workers=4,
    epochs=20,
    seed=42
)

print("Vocabulary size:", len(w2v.wv))
print(w2v.wv.most_similar("shirt", topn=5))  # sanity check, use any word in your vocab

Vocabulary size: 22926
[('polo', 0.9608508348464966), ('kerah', 0.9391984343528748), ('poloshirt', 0.9337562918663025), ('berkerah', 0.9304197430610657), ('panjang/kaos', 0.9304147958755493)]


In [10]:
def title_to_vec(tokens, model):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if len(vecs) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

X_w2v = np.vstack([title_to_vec(t, w2v) for t in tokenized])
print(X_w2v.shape)

(34250, 100)


In [21]:
def pair_cosine(X, df):
    A = X[df["index_1"].values]
    B = X[df["index_2"].values]
    num = (A * B).sum(axis=1)
    den = np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1)
    return num / np.maximum(den, 1e-9)

pairs_df["sim_w2v"] = pair_cosine(X_w2v, pairs_df)
second_pairs["sim_w2v"] = pair_cosine(X_w2v, second_pairs)

In [24]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix

thresholds = np.arange(0.5, 1.0, 0.02)
rows = []
for th in thresholds:
    pred = (pairs_df["sim_w2v"] >= th).astype(int)
    rows.append({
        "threshold": round(th, 2),
        "precision": precision_score(pairs_df["match"], pred, zero_division=0),
        "recall": recall_score(pairs_df["match"], pred),
        "f1": f1_score(pairs_df["match"], pred),
    })
th_df = pd.DataFrame(rows)
print(th_df.to_string(index=False))

best_th = th_df.loc[th_df["f1"].idxmax(), "threshold"]
print("Best threshold (tuning set):", best_th)

 threshold  precision   recall       f1
      0.50   0.786564 0.967867 0.867847
      0.52   0.816814 0.957333 0.881509
      0.54   0.843687 0.942033 0.890152
      0.56   0.869533 0.927067 0.897378
      0.58   0.893147 0.911933 0.902443
      0.60   0.912812 0.895833 0.904243
      0.62   0.930745 0.876700 0.902915
      0.64   0.945826 0.857233 0.899353
      0.66   0.956772 0.835900 0.892261
      0.68   0.965831 0.811233 0.881807
      0.70   0.973983 0.783667 0.868521
      0.72   0.980259 0.749800 0.849680
      0.74   0.985892 0.717433 0.830507
      0.76   0.990398 0.680733 0.806875
      0.78   0.993360 0.643333 0.780918
      0.80   0.995193 0.600433 0.748981
      0.82   0.996543 0.557267 0.714811
      0.84   0.997733 0.513500 0.678037
      0.86   0.998495 0.464567 0.634105
      0.88   0.998936 0.406967 0.578324
      0.90   0.999319 0.342367 0.510005
      0.92   0.999631 0.271167 0.426609
      0.94   0.999660 0.196233 0.328067
      0.96   1.000000 0.123900 0.220482


In [25]:
second_pairs["pred_w2v"] = (second_pairs["sim_w2v"] >= best_th).astype(int)

y_true = second_pairs["match"]
y_pred = second_pairs["pred_w2v"]

print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall   :", recall_score(y_true, y_pred))
print("F1-score :", f1_score(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))

Accuracy : 0.9155
Precision: 0.0
Recall   : 0.0
F1-score : 0.0
[[9155  845]
 [   0    0]]


c:\Users\krish\anaconda3\envs\pytorch_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [26]:
import random
random.seed(7)

# pairs already used (store both orders so the check is reliable)
used = set()
for a, b in zip(pairs_df["index_1"], pairs_df["index_2"]):
    used.add((a, b)); used.add((b, a))

multi_groups = [idx for idx in label_groups.values() if len(idx) >= 2]

N = 5000  # per class

# positives: random pair from within a group
eval_pos = set()
while len(eval_pos) < N:
    g = random.choice(multi_groups)
    i, j = random.sample(g, 2)
    if (i, j) not in used and (j, i) not in used:
        eval_pos.add((min(i, j), max(i, j)))

# negatives: random pairs from different groups
eval_neg = set()
n = len(y)
while len(eval_neg) < N:
    i, j = random.sample(range(n), 2)
    if y.iloc[i] != y.iloc[j] and (i, j) not in used:
        eval_neg.add((min(i, j), max(i, j)))

eval_pairs = pd.DataFrame(
    [(i, j, 1) for i, j in eval_pos] + [(i, j, 0) for i, j in eval_neg],
    columns=["index_1", "index_2", "match"]
).sample(frac=1, random_state=42).reset_index(drop=True)

print(eval_pairs["match"].value_counts())

match
0    5000
1    5000
Name: count, dtype: int64


#### Comparision of recall and word2vec

In [30]:
# Word2Vec
eval_pairs["sim_w2v"] = pair_cosine(X_w2v, eval_pairs)
eval_pairs["pred_w2v"] = (eval_pairs["sim_w2v"] >= best_th).astype(int)

# TF-IDF baseline (vectorized, much faster than your apply)
eval_pairs["sim_tfidf"] = pair_cosine(X_tfidf, eval_pairs)
eval_pairs["pred_tfidf"] = (eval_pairs["sim_tfidf"] >= 0.5).astype(int)

for name in ["tfidf", "w2v"]:
    yt, yp = eval_pairs["match"], eval_pairs[f"pred_{name}"]
    print(name,
          "| P:", round(precision_score(yt, yp, zero_division=0), 3),
          "| R:", round(recall_score(yt, yp), 3),
          "| F1:", round(f1_score(yt, yp), 3))
    print(confusion_matrix(yt, yp))

tfidf | P: 1.0 | R: 0.65 | F1: 0.788
[[4999    1]
 [1750 3250]]
w2v | P: 0.919 | R: 0.86 | F1: 0.889
[[4623  377]
 [ 698 4302]]
